In [1]:
import sympy
import numpy as np
import itertools


from sympy import *
from sympy.combinatorics import Permutation

In [2]:
t, t1, t2, tau1, tau2, theta, mu, kappa, alpha, beta, s1, s2, s3 = vrs = symbols("t, t1, t2, tau1, tau2, theta, mu, kappa, alpha, beta, s1, s2, s3")

# for vr in vrs:
#     assume(vr, "real")
#     assume(vr > 0)

In [4]:
# AGCT version:
Qagct = Matrix(
    [
        [-kappa-2,   kappa,       1,       1],
        [   kappa,-kappa-2,       1,       1],
        [       1,       1,-kappa-2,   kappa],
        [       1,       1,   kappa,-kappa-2],
    ]
)


# ACGT version:
P = Matrix(PermutationMatrix(Permutation([0,2,1,3])))
# P = ZeroMatrix(4,4)
# P[0,0] = 1
# P[1,2] = 1
# P[2,1] = 1
# P[3,3] = 1

Qacgt = P * Qagct * P

In [5]:
Qacgt

Matrix([
[-kappa - 2,          1,      kappa,          1],
[         1, -kappa - 2,          1,      kappa],
[     kappa,          1, -kappa - 2,          1],
[         1,      kappa,          1, -kappa - 2]])

In [6]:
-sum(Qacgt.diagonal())/4

kappa + 2

In [7]:
Pacgt = exp(Qacgt*t)

Pacgt

Matrix([
[ exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4,                            1/4 - exp(-4*t)/4, -exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4,                            1/4 - exp(-4*t)/4],
[                           1/4 - exp(-4*t)/4,  exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4,                            1/4 - exp(-4*t)/4, -exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4],
[-exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4,                            1/4 - exp(-4*t)/4,  exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4,                            1/4 - exp(-4*t)/4],
[                           1/4 - exp(-4*t)/4, -exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4,                            1/4 - exp(-4*t)/4,  exp(-2*kappa*t - 2*t)/2 + 1/4 + exp(-4*t)/4]])

In [8]:
densities = np.einsum("ni,nj,nk->ijk", Pacgt.subs(t,s1), Pacgt.subs(t,s2), Pacgt.subs(t,s3)) / 4

# integrals

In [11]:
data = [0] * (4**3)

sps = MutableDenseNDimArray(data,shape=(4,4,4))

In [12]:
# S0 ((a,b),c)
for i, j, k in itertools.product(range(4), repeat=3):
    print(i,j,k)
    sps[i,j,k] += (
        integrate(
            integrate(
                (
                    # T0 congruent ((a,b),c)
                    integrate(
                        integrate(
                            densities[i,j,k].subs({s1:t1,s2:t1,s3:2*t2-t1}) * exp( -2/theta*(t1 - tau1)) * exp( -2/theta * (t2-tau2))* (2/theta)**22,
                            (t1, tau1, tau2)
                        ),
                        (t2, tau2, oo)
                    ) +
                    # T1 deep congruent ((a,b),c)
                    integrate(
                        integrate(
                            densities[i,j,k].subs({s1:t1,s2:t1,s3:2*t2-t1}) * exp( -2/theta*(t1 - tau2)) * exp( -2/theta * (t2-tau2))* (2/theta)**2,
                            (t1, tau2, t2)
                        ),
                        (t2, tau2, oo)
                    ) +
                    # T2 deep discordant: ((a,c),b) 
                    integrate(
                        integrate(
                            densities[i,j,k].subs({s1:t1,s2:2*t2-t1,s3:t1}) * exp( -2/theta*(t1 - tau2)) * exp( -2/theta * (t2-tau2))* (2/theta)**2,
                            (t1, tau2, t2)
                        ),
                        (t2, tau2, oo)
                    ) +
                    # T3 deep discordant ((b,c),a)
                    integrate(
                        integrate(
                            densities[i,j,k]({s1:2*t2-t1,s2:t1,s3:t1}) * exp( -2/theta*(t1 - tau2)) * exp( -2/theta * (t2-tau2)) * (2/theta)**2,
                            (t1, tau2, t2)
                        ),
                        (t2, tau2, oo)
                    )
                ) * exp( -2/theta * tau1 ) * exp( -2/theta *( tau2-tau1)) * 2/theta^2,
                (tau1, 0, tau2)
            ),
            (tau2, 0, oo)
        )
    )


0 0 0


KeyboardInterrupt: 